In [4]:
from pathlib import Path
import pandas as pd
import json

raw_data = Path("../data/raw/")

data_list = []

counter = 0
for file in raw_data.iterdir():
    with open(file) as f:
        for row in f:
            data = json.loads(row)
            data_list.append(data)
            counter+= 1

            if counter == 100000:
                break
    if counter == 100000:
        break

In [6]:
df = pd.json_normalize(data_list)

print(df.columns)

Index(['$type', 'cid', 'collection', 'did', 'operation', 'rev', 'rkey', 'seq',
       'time', 'record.$type',
       ...
       'record.social.rhize.commentSecretVersion',
       'record.social.rhize.quoteLayout.hasText',
       'record.social.rhize.quoteLayout.media.aspect.height',
       'record.social.rhize.quoteLayout.media.aspect.width',
       'record.social.rhize.quoteLayout.media.kind', 'record.',
       'record.uk.skyblur.post.uri', 'record.uk.skyblur.post.visibility',
       'record.embed.id', 'record.embed.type'],
      dtype='str', length=130)


In [7]:
df_filtered = df[["record.text","record.langs"]]

print("General statistics")
print(df_filtered.columns)
print(df_filtered.head())
print(df_filtered.count())
print(df_filtered["record.langs"].explode().value_counts())
print(df_filtered["record.langs"].isnull().sum())
print("=========")

print("Words distribution")
df_filtered["words_count"] = df_filtered['record.text'].str.split().str.len()
print(df_filtered["words_count"].describe())
print("=========")

print("Duplicated texts")
df_filtered["normalized_text"] = df_filtered['record.text'].str.lower().str.strip()
print(df_filtered["normalized_text"].duplicated().sum())
print("=========")

print("Noise")
print(df_filtered["record.text"].str.contains("@").mean())
print(df_filtered["record.text"].str.contains("#").mean())
print(df_filtered["record.text"].str.contains("http").mean())
print("=========")

General statistics
Index(['record.text', 'record.langs'], dtype='str')
                                         record.text record.langs
0                               Arby’s steak nuggets         [en]
1               Katatonia \nDeliberation \n#MusicSky         [en]
2  When we talk about coddling kids this is the t...         [en]
3                             Joo, näyttää hienolta.         [fi]
4  tysm my dearest shellie 🥺🫶 extra funds for the...         [en]
record.text     100000
record.langs     89946
dtype: int64
record.langs
en    57525
fr    13050
nl     4735
ja     4427
de     2526
      ...  
pk        1
ig        1
sg        1
za        1
vn        1
Name: count, Length: 108, dtype: int64
10054
Words distribution
count    100000.000000
mean         15.955570
std          14.889859
min           0.000000
25%           4.000000
50%          11.000000
75%          23.000000
max          82.000000
Name: words_count, dtype: float64
Duplicated texts
10358
Noise
0.05778
0.08347
0.